In [4]:
import json

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from pathlib import Path
from tqdm import tqdm

In [5]:
with open("data/db_docs.json", "r") as f:
    loaded_docs_json = json.load(f)

chunks = [
    Document(page_content=doc["text"], metadata={
        'lang': doc['lang'],
        'tr_lang': doc['tr_lang'],
        'tr_text': doc['tr_text'],
        'file_id': doc['file_id']
    })
    for doc in loaded_docs_json
]

print(f"Restored chunks from json: {len(chunks):_}")

Restored chunks from json: 371_476


In [6]:
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

def get_batches(chunks, batch_size=1024):
    for i in range(0, len(chunks), batch_size):
        yield chunks[i:i+batch_size]

In [19]:
db_name = 'chroma_db'
db_path = Path(db_name)

vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

total_docs = vectorstore._collection.count()
if total_docs == 0:
    for batch in tqdm(get_batches(chunks), desc="Adding documents", unit="batch"):
        vectorstore.add_documents(batch)
    print(f"Added {vectorstore._collection.count()} documents.")

total_docs = vectorstore._collection.count()
print(f"Found {total_docs} documents in Chroma vectorstore `{db_name}`")

Adding documents: 363batch [58:50,  9.72s/batch]

Added 371476 documents.
Found 371476 documents in Chroma vectorstore `chroma_db`
